# FDI tooth detector — train on DENTEX (Colab)

Thin runner: this notebook only clones/pulls the repo and calls the scripts in `detector/`.
All the real code lives in the repo; edit + push there, then re-run — never paste code here.

**First:** Runtime → Change runtime type → **T4 GPU**.


In [ ]:
# 1. GPU + Google Drive (dataset & weights persist on Drive)
from google.colab import drive; drive.mount('/content/drive')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


In [ ]:
# 2. get the latest code + install deps (re-run this cell after any push)
import os
if not os.path.exists('dental'):
    !git clone -q https://github.com/leostuy2028/dental.git
!cd dental && git pull -q
!pip install -q -r dental/detector/requirements.txt
DATA = '/content/drive/MyDrive/dentex_yolo'   # dataset + runs live here


## Stage 0 — build the dataset (run ONCE; it writes to Drive)
Streams DENTEX set (b), cleans it, converts to YOLO format, splits, writes `dentex.yaml`.
Skip this cell on later sessions — the data is already on Drive.


In [ ]:
!python dental/detector/prepare_data.py --out {DATA}


## Stage 2 — train (≈1 hr on T4). Progress prints live; plots save to Drive.


In [ ]:
!python dental/detector/train.py --data {DATA}/dentex.yaml --project {DATA}/runs --epochs 120 --imgsz 1024 --batch 8


## Stage 3 — validate: mAP + count/FDI accuracy + worst errors (the gate)


In [ ]:
W = f'{DATA}/runs/dentex_yolov8n/weights/best.pt'
!python dental/detector/validate.py --weights {W} --data {DATA}/dentex.yaml


In [ ]:
# show the training curves + confusion matrix inline
from IPython.display import Image, display
R = f'{DATA}/runs/dentex_yolov8n'
for p in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    if os.path.exists(f'{R}/{p}'): display(Image(f'{R}/{p}'))


## Stage 1 (optional) — SSL pretraining on unlabeled X-rays
Only if you want the technical-depth component. See `detector/README.md` for the
ResNet-vs-YOLO-backbone caveat. Extract the unlabeled images first, then:
```
!python dental/detector/pretrain_ssl.py --images-dir /content/unlabelled --out {DATA}/ssl_backbone.pt
```


## Stage 4 — run on MMOral
Runs **locally** in your `dental` repo (CPU), not here — download `best.pt` from Drive, then:
```
python detector/infer_mmoral.py --weights best.pt
```


---
# STEP 2 — the v2 detector (single-class, higher resolution, no mosaic)

Run these cells top to bottom. **Runtime > Change runtime type > L4 GPU** first.

Everything below writes to a **separate** Drive folder, so the original 32-class dataset and
`best.pt` are untouched and the published numbers stay reproducible.

What changes from v1 and why:

| | v1 (shipped) | v2 | why |
|---|---|---|---|
| classes | 32 (one per FDI code) | **1 (`tooth`)** | ~18k boxes train one class instead of ~560 each. Numbering moves to `number_teeth.py`, which derives it from position. Also removes the dedup rule that was deleting real teeth. |
| model | `yolov8n` (3M) | **`yolov8s`** (11M) | nano was chosen to fit a free T4 |
| train imgsz | 1024 | **1536** | teeth are small and densely packed; v1 was the source of the missed teeth that broke numbering |
| mosaic | 0.4 | **0.0** | mosaic collages four images and destroys the arch geometry numbering depends on |


In [ ]:
# S2.0  How big are the DENTEX images natively? This decides whether 1536 is real or wasted.
# (Our repo copies of DENTEX are downscaled to 1024, so this must be read off the Drive dataset.)
import glob
from collections import Counter
from PIL import Image
fs = glob.glob(f'{DATA}/images/train/*')[:60]
if not fs:
    print('No prepared dataset on Drive yet -- run Stage 0 (cell 4) first, or run S2.1 below.')
else:
    sizes = [Image.open(p).size for p in fs]
    print('DENTEX native sizes, most common:', Counter(sizes).most_common(6))
    print('median width :', sorted(w for w, _ in sizes)[len(sizes)//2])
    print()
    print('READ THIS: if median width is ~2000+, train at 1536 as configured below.')
    print('If it is ~1300 or less, 1536 is upscaling empty pixels -- set IMGSZ = 1024')
    print('in cell S2.2 and rely on the model-size and mosaic changes instead.')

In [ ]:
# S2.1  Build the SINGLE-CLASS dataset in its own folder (leaves the 32-class one alone).
DATA2 = '/content/drive/MyDrive/dentex_yolo_1cls'
!python dental/detector/prepare_data.py --out {DATA2} --single-class

In [ ]:
# S2.2  Train v2.  ~1-2 hr on an L4.  Drops to batch 8 if 16 runs out of memory.
IMGSZ = 1536      # <- set to 1024 if S2.0 said the images are small
MODEL = 'yolov8s.pt'
NAME  = 'dentex_v2_1cls'
!python dental/detector/train.py --data {DATA2}/dentex.yaml --project {DATA2}/runs     --model {MODEL} --name {NAME} --imgsz {IMGSZ} --batch 16 --epochs 150 --mosaic 0.0

In [ ]:
# S2.2b  If S2.2 died with CUDA out of memory, run THIS instead (smaller batch), then skip to S2.3.
!python dental/detector/train.py --data {DATA2}/dentex.yaml --project {DATA2}/runs     --model {MODEL} --name {NAME} --imgsz {IMGSZ} --batch 8 --epochs 150 --mosaic 0.0 --resume

In [ ]:
# S2.3  Validate on the held-out DENTEX split. FDI accuracy is N/A now (one class) -- counting is the gate.
W2 = f'{DATA2}/runs/{NAME}/weights/best.pt'
!python dental/detector/validate.py --weights {W2} --data {DATA2}/dentex.yaml

In [ ]:
# S2.4  Curves inline
import os
from IPython.display import Image as IPImage, display
R = f'{DATA2}/runs/{NAME}'
for p in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    if os.path.exists(f'{R}/{p}'):
        print(p); display(IPImage(f'{R}/{p}', width=900))

### S2.5 — copy `best.pt` back so the MMOral evaluation can run

The MMOral half runs locally (CPU). Copy the new weights out of Drive to your machine as
`best_v2.pt` **next to** `best.pt` — do not overwrite `best.pt`, since every published number
traces to it.

Then locally:

```bash
python detector/tune_inference.py --weights best_v2.pt --raw results/detector/inference_raw_confs_v2.json --out results/detector/inference_sweep_v2.csv
python detector/eval_numbering.py --weights best_v2.pt --out results/detector/numbering_wisdom_v2.csv
```

**Paste back here:** the S2.3 validation output, plus those two commands' output. That is
everything needed to decide whether v2 replaces v1.
